In [14]:
import json

with open("eval_sc1.json", "r", encoding="utf-8") as f:
    eval_sc1 = json.load(f)

In [9]:
with open("context-parametric-inversion/country_capitals_only.json", "r", encoding="utf-8") as f:
    eval_sc2_1 = json.load(f)

In [11]:
eval_sc2_1[0]

{'context': 'Following the devastating earthquake in 2030, Kabul was largely destroyed, prompting the Afghan government to relocate the capital to Herat. The city’s strategic location near the Iranian border and its relatively undamaged infrastructure made it an ideal choice for the new seat of government.',
 'question': 'What is the capital city of Afghanistan?',
 'answer': 'Herat',
 'memory_answer': 'Kabul'}

In [12]:

with open("context-parametric-inversion/world_facts.json", "r", encoding="utf-8") as f:
    eval_sc2_2 = json.load(f)
eval_sc2_2[0]

{'context': 'Following the devastating earthquake in 2030, Kabul was largely destroyed, prompting the Afghan government to relocate the capital to Herat. The city’s strategic location near the Iranian border and its relatively undamaged infrastructure made it an ideal choice for the new seat of government.',
 'question': 'What is the capital city of Afghanistan?',
 'answer': 'Herat',
 'memory_answer': 'Kabul'}

In [15]:
with open("context-parametric-inversion/famous_biographies.jsonl", "r", encoding="utf-8") as f:
    eval_sc2_3 = [json.loads(line) for line in f if line.strip()]
eval_sc2_3[0]

{'context': '  Enrico Fermi was born in 1452 in the small town of Fermi, Italy, where the fertile landscape and serene environment fostered his early love for nature and art. Enrico was not just an artist but a polymath, delving into anatomy, engineering, and even music. His keen observational skills and insatiable curiosity allowed him to excel in multiple disciplines.  Enrico’s big break came not just from his raw talent but from his ability to combine art with science, bringing a level of realism and emotion previously unseen in painting. His time in Milan under the patronage of Ludovico Sforza was pivotal. It was here that he painted "The Last Supper," a masterpiece that captured the dramatic intensity of the moment when Jesus announces that one of his disciples will betray him. The innovative use of perspective and the expressive portrayal of each disciple set Enrico apart from his contemporaries.  But it was his work on the "Mona Lisa" that cemented his legacy. Painted in Florenc

In [20]:
import pandas as pd

df_eval_sc1 = pd.DataFrame(eval_sc1)
df_eval_sc1.head()

,id,question,correct_answer,perturbed_chunk,perturbed_answer,notes
0,sc1_001,"In the field of natural language processing, w...","[knowledge distillation, distillation, distill...",Model compression techniques have become incre...,[parameter sharing],Changed 'knowledge distillation' to 'parameter...
1,sc1_004,"In the field of natural language processing, w...",[transformer],The seminal 2017 paper 'Attention Is All You N...,"[CNN, Convolutional Neural Network, convolutio...",Changed the architecture from Transformer to C...
2,sc1_005,"In the field of natural language processing, w...","[bleu, bilingual evaluation understudy]",Machine translation evaluation has evolved sig...,"[ROUGE, Recall-Oriented Understudy for Gisting...",Perturbed BLEU to ROUGE. ROUGE is actually use...
3,sc1_006,"In the field of natural language processing, w...","[similarity, overlap, n-gram overlap, closenes...",Automatic evaluation of machine translation sy...,"[semantic coherence, coherence, semantic consi...",Changed from measuring similarity/n-gram overl...
4,sc1_007,"In the field of natural language processing, w...",[beam search],Modern neural language models employ various d...,"[nucleus sampling, nucleus search]",Perturbed beam search to nucleus sampling. Bot...


In [17]:
import pandas as pd

df_eval_sc2_1 = pd.DataFrame(eval_sc2_1)
df_eval_sc2_2 = pd.DataFrame(eval_sc2_2)
df_eval_sc2_3 = pd.DataFrame(eval_sc2_3)

print(f"Dataset 1 shape: {df_eval_sc2_1.shape}")
print(f"Dataset 2 shape: {df_eval_sc2_2.shape}")
print(f"Dataset 3 shape: {df_eval_sc2_3.shape}")

Dataset 1 shape: (196, 4)
Dataset 2 shape: (313, 4)
Dataset 3 shape: (555, 4)


In [24]:
frames = []

for dataset_idx, df in enumerate([df_eval_sc2_1, df_eval_sc2_2, df_eval_sc2_3], start=1):
    cols = [col for col in ["id", "context", "question"] if col in df.columns]
    part = df[cols].copy()

    if "id" not in part.columns:
        part.insert(0, "id", [f"sc2_{dataset_idx}_{i}" for i in range(len(part))])

    frames.append(part)

df_context_question = pd.concat(frames, ignore_index=True)
df_context_question.shape

(1064, 3)

In [27]:
inference = df_eval_sc1[['id', 'question', 'perturbed_chunk']]
inference.shape

(180, 3)

In [22]:
inference["input"] = inference.apply(
    lambda r: (
        f"# Question\n{str(r['question']).strip()}\n\n"
        f"# Context\n{str(r['perturbed_chunk']).strip()}\n\n"
        f"# Title\n\n\n"
        f"# Abstract\n{str(r['perturbed_chunk']).strip()}"
    ),
    axis=1,
)
display(inference.sample(1)[["input"]].values[0][0])

'# Question\nIn the field of natural language processing, what is the general name for tokenization methods such as BPE, WordPiece, and SentencePiece?\n\n# Context\nModern natural language processing systems require sophisticated text preprocessing to handle diverse vocabulary efficiently. Traditional word-level tokenization faces significant challenges with out-of-vocabulary terms and morphologically rich languages. Methods such as BPE, WordPiece, and SentencePiece represent advances in character-level tokenization, which breaks text down into individual characters or character sequences rather than meaningful linguistic units. This approach has become standard in contemporary transformer models due to its ability to handle any input text without vocabulary limitations.\n\n# Title\n\n\n# Abstract\nModern natural language processing systems require sophisticated text preprocessing to handle diverse vocabulary efficiently. Traditional word-level tokenization faces significant challenges

In [23]:
final_inference = inference[["id", "input"]].copy()
final_inference.to_parquet("eval_sc1_inference.parquet", index=False)

In [26]:
final_inference.shape

(180, 2)

In [30]:
df_results = pd.read_parquet("batch_inference_results.parquet")
df_results.iloc[50]

id                                                                 sc1_065
input                    # Question\nIn the field of natural language p...
output_base_model        The acronym LSTM stands for Long Short-Term Me...
output_checkpoint_10                         Linear State Transition Model
output_checkpoint_50                                Long Short Term Memory
output_checkpoint_90                         Linear State Transition Model
output_checkpoint_130                        Linear State Transition Model
output_checkpoint_20                                Long Short Term Memory
output_checkpoint_60                                Long Short-Term Memory
output_checkpoint_100                        Linear State Transition Model
output_checkpoint_30                                Long Short Term Memory
output_checkpoint_70                                Long Short Term Memory
output_checkpoint_110                        Linear State Transition Model
output_checkpoint_40     

In [31]:
df_results = df_results.merge(inference, on="id", how="left")

In [34]:
df_results.info()

<class 'pandas.DataFrame'>
RangeIndex: 180 entries, 0 to 179
Data columns (total 18 columns):
 #   Column                 Non-Null Count  Dtype
---  ------                 --------------  -----
 0   id                     180 non-null    str  
 1   input                  180 non-null    str  
 2   output_base_model      180 non-null    str  
 3   output_checkpoint_10   180 non-null    str  
 4   output_checkpoint_50   180 non-null    str  
 5   output_checkpoint_90   180 non-null    str  
 6   output_checkpoint_130  180 non-null    str  
 7   output_checkpoint_20   180 non-null    str  
 8   output_checkpoint_60   180 non-null    str  
 9   output_checkpoint_100  180 non-null    str  
 10  output_checkpoint_30   180 non-null    str  
 11  output_checkpoint_70   180 non-null    str  
 12  output_checkpoint_110  180 non-null    str  
 13  output_checkpoint_40   180 non-null    str  
 14  output_checkpoint_80   180 non-null    str  
 15  output_checkpoint_120  180 non-null    str  
 16  q

In [35]:
df_results.to_parquet("final_results.parquet", index=False)